In [1]:
import os

target_folder = "https://github.com/eoconn25/CS6423_knowledge_distillation_project" 
path = os.path.join(os.getcwd(), target_folder)

if os.path.exists(path):
    os.chdir(path)

# should match the folder you cloned into
print(f"Current working directory: {os.getcwd()}")

Current working directory: c:\Users\fiach\Documents\Code\msc\scalable\CS6423_knowledge_distillation_project


Use this notebook to download the dataset used for training and validation for the radimage models

This notebook pulls the dataset from [here](https://huggingface.co/datasets/raidium/RadImageNet-VQA/viewer/alignment/train?row=0)

It then uses the metdata column to construct labels for the images. The images are stored in `data/test_images` and the labels are stored in `data/labels.csv`

In [2]:
import os

CONFIG = "benchmark"
SAVE_DIR = "data/test_images"
os.makedirs(SAVE_DIR, exist_ok=True)

The next cell will help you to login and authenticate with hugging face which is necessary for downloading the dataset

1. Request Access (Browser)
- Go to the [dataset page](https://huggingface.co/datasets/raidium/RadImageNet-VQA) on Hugging Face
- Log in to your account
- Click the "Agree and access repository" (or similar) button at the top of the dataset card

2. Generate Access Token
- Navigate to your [Settings > Access Tokens](https://huggingface.co/settings/tokens)
- Click "New token"
- Give it a name and set the role to "Read"
- Copy the token (it starts with hf_...)

3. Paste the token into the `login()` function string below

In [ ]:
from huggingface_hub import login
login("hf_your_token_here")

c:\Users\fiach\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset
from tqdm.auto import tqdm

print(f"Downloading RadImageNet-VQA ({CONFIG} config)...")

raw_dataset = load_dataset("raidium/RadImageNet-VQA", CONFIG)

ds = raw_dataset["test"]
print(f"Dataset loaded successfully. Found {len(ds)} samples.")

Dataset loaded successfully. Found 9000 samples.


In [5]:
seen_images = set()

labels_path = "data/labels.csv"

with open(labels_path, "w", encoding='utf-8') as f:
    f.write("filename,pathology,modality,location,label\n")
    
    for i in tqdm(range(len(ds)), desc="Saving unique images"):
        row = ds[i]
        
        if not isinstance(row, dict):
            continue
            
        # unique ID for the image (one image has ~9 questions)
        img_id = row.get("id", i) 
        
        if img_id not in seen_images:
            img_filename = f"test_{img_id}.png"
            img_path = os.path.join(SAVE_DIR, img_filename)
            
            # if images already exist skip saving them
            if not os.path.exists(img_path):
                row["image"].save(img_path)
                
            meta = row.get("metadata", {})
            pathology = meta.get("pathology", "unknown")
            modality = meta.get("modality", "unknown")
            location = meta.get("location", "unknown")
            label = f"{location}_{pathology}".replace(" ", "_")
                
            f.write(f"{img_filename},{pathology},{modality},{location},{label}\n")
            seen_images.add(img_id)

print(f"\nProcessing Complete!")
print(f"Total Unique Images: {len(seen_images)}")
print(f"Metadata saved to: {labels_path}")

Saving unique images: 100%|██████████| 9000/9000 [00:10<00:00, 822.29it/s]


Processing Complete!
Total Unique Images: 9000
Metadata saved to: data/labels.csv
